# Numeric Feature Inspection — Engelen-cleaned CIC-IDS2017

**Notebook 02 of EDA series**
**Author:** Bradly Morein (UP2088701)
**Date:** 27 April 2026

## Purpose

Audit numeric features for data integrity issues that will affect 
preprocessing decisions:

1. **NaN and Inf values** — count per feature, per day. Required because 
   scikit-learn and TensorFlow both throw on NaN/Inf inputs.
2. **Constant or near-constant features** — identify features carrying 
   zero or negligible information. Drop candidates.
3. **Protocol=0 artefact rows** — non-IP frames that CICFlowMeter 
   recorded as flow records (observed in notebook 01). Count and 
   characterise their distribution; decide handling.

Each finding produces a numerical answer that feeds directly into 
Methodology chapter §4.3 (Pre-processing Pipeline) and the 
preprocessing code in `src/preprocessing/`.

In [1]:
"""Imports and notebook-wide settings."""
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / ".git").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
PARQUET_DIR = REPO_ROOT / "data" / "interim" / "cicids2017-engelen"

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 100)

day_files = sorted(PARQUET_DIR.glob("*.parquet"))
print(f"Repo root: {REPO_ROOT}")
print(f"Files to inspect: {len(day_files)}")
for p in day_files:
    print(f"  {p.name}")

Repo root: /home/brad/projects/dissertation
Files to inspect: 5
  Friday-WorkingHours.parquet
  Monday-WorkingHours.parquet
  Thursday-WorkingHours.parquet
  Tuesday-WorkingHours.parquet
  Wednesday-WorkingHours.parquet


## NaN and Inf audit

Both `numpy.nan` and `numpy.inf` (positive or negative) cause failures in 
scikit-learn and TensorFlow. We need to know:

- **Total count** per feature across the whole dataset.
- **Per-day distribution** — are issues concentrated on certain days, 
  or evenly spread?
- **Which feature families** are affected — flow rates (Bytes/s, Packets/s) 
  are the usual suspects because the duration denominator can be zero.

In [2]:
"""Audit NaN and Inf across all five days."""

# Container for per-day counts
nan_per_day = {}
inf_per_day = {}

print("Counting NaN and Inf per file...\n")

for path in day_files:
    day = path.stem.replace("-WorkingHours", "")
    df = pd.read_parquet(path)
    
    # Numeric columns only — strings (Flow ID etc.) cannot have NaN/Inf
    numeric = df.select_dtypes(include="number")
    
    nan_counts = numeric.isna().sum()
    # np.isinf returns True for both +inf and -inf
    inf_counts = numeric.apply(lambda col: np.isinf(col).sum())
    
    nan_per_day[day] = nan_counts
    inf_per_day[day] = inf_counts
    
    n_nan = nan_counts.sum()
    n_inf = inf_counts.sum()
    print(f"  {day:10s}: {len(df):>7,} rows | NaN cells: {n_nan:>6,} | Inf cells: {n_inf:>6,}")

# Aggregate across days into a wide table
nan_table = pd.DataFrame(nan_per_day).fillna(0).astype(int)
inf_table = pd.DataFrame(inf_per_day).fillna(0).astype(int)

# Combined summary: features that have either NaN or Inf anywhere
nan_table["total_nan"] = nan_table.sum(axis=1)
inf_table["total_inf"] = inf_table.sum(axis=1)

problem_features = nan_table.index[
    (nan_table["total_nan"] > 0) | (inf_table["total_inf"] > 0)
].tolist()

print(f"\n{'='*60}")
print(f"Features with NaN or Inf anywhere: {len(problem_features)}")
print(f"{'='*60}")

Counting NaN and Inf per file...

  Friday    : 547,915 rows | NaN cells:    506 | Inf cells:    398
  Monday    : 371,749 rows | NaN cells:    188 | Inf cells:    120
  Thursday  : 362,368 rows | NaN cells:    501 | Inf cells:    321
  Tuesday   : 322,003 rows | NaN cells:     95 | Inf cells:     43
  Wednesday : 496,779 rows | NaN cells:    320 | Inf cells:    102

Features with NaN or Inf anywhere: 6


In [3]:
"""Identify exactly which features have NaN/Inf and how concentrated they are."""

# Combine NaN and Inf into one summary table
issue_summary = pd.DataFrame({
    "nan_total":      nan_table["total_nan"],
    "inf_total":      inf_table["total_inf"],
})
issue_summary["combined_total"] = issue_summary.sum(axis=1)
issue_summary = issue_summary[issue_summary["combined_total"] > 0].sort_values(
    "combined_total", ascending=False
)

print(f"{'='*70}")
print(f"Features with NaN or Inf — full breakdown")
print(f"{'='*70}\n")
print(issue_summary.to_string())

print(f"\n{'='*70}")
print(f"Per-day distribution for the worst-affected feature(s)")
print(f"{'='*70}")

# For each problem feature, show the per-day NaN and Inf counts
for feature in issue_summary.index:
    print(f"\n{feature}:")
    print(f"  NaN per day: ", end="")
    for day in nan_per_day.keys():
        print(f"{day[:3]}={nan_per_day[day][feature]:>4}  ", end="")
    print()
    print(f"  Inf per day: ", end="")
    for day in inf_per_day.keys():
        print(f"{day[:3]}={inf_per_day[day][feature]:>4}  ", end="")
    print()

Features with NaN or Inf — full breakdown

                nan_total  inf_total  combined_total
Flow Bytes/s          602        191             793
Flow Packets/s          0        793             793
Flow IAT Mean         252          0             252
Flow IAT Std          252          0             252
Flow IAT Max          252          0             252
Flow IAT Min          252          0             252

Per-day distribution for the worst-affected feature(s)

Flow Bytes/s:
  NaN per day: Fri= 298  Mon=   0  Thu= 261  Tue=   3  Wed=  40  
  Inf per day: Fri=  50  Mon=  60  Thu=  30  Tue=  20  Wed=  31  

Flow Packets/s:
  NaN per day: Fri=   0  Mon=   0  Thu=   0  Tue=   0  Wed=   0  
  Inf per day: Fri= 348  Mon=  60  Thu= 291  Tue=  23  Wed=  71  

Flow IAT Mean:
  NaN per day: Fri=  52  Mon=  47  Thu=  60  Tue=  23  Wed=  70  
  Inf per day: Fri=   0  Mon=   0  Thu=   0  Tue=   0  Wed=   0  

Flow IAT Std:
  NaN per day: Fri=  52  Mon=  47  Thu=  60  Tue=  23  Wed=  70  
  Inf

In [4]:
"""Verify the structural hypotheses about the bad rows."""

print("Hypothesis 1: Flow Bytes/s and Flow Packets/s break on the same rows")
print("              (both denominators are flow_duration == 0)")
print("="*70)

# Pick Friday because it has the most problems
df = pd.read_parquet(day_files[0])  # day_files[0] is Friday
print(f"\nLoaded {day_files[0].name}: {len(df):,} rows\n")

bytes_bad = df["Flow Bytes/s"].isna() | np.isinf(df["Flow Bytes/s"])
pkts_bad  = df["Flow Packets/s"].isna() | np.isinf(df["Flow Packets/s"])

print(f"Rows with bad Flow Bytes/s:   {bytes_bad.sum():>4}")
print(f"Rows with bad Flow Packets/s: {pkts_bad.sum():>4}")
print(f"Rows with EITHER bad:         {(bytes_bad | pkts_bad).sum():>4}")
print(f"Rows with BOTH bad:           {(bytes_bad & pkts_bad).sum():>4}")
print(f"Mismatch (XOR):               {(bytes_bad ^ pkts_bad).sum():>4}")

print(f"\nSample of bad rows on Friday — Flow Duration values:")
print(df.loc[bytes_bad | pkts_bad, "Flow Duration"].describe())

print("\n")
print("Hypothesis 2: Flow IAT NaN rows are single-packet flows")
print("              (i.e. Total Fwd Packet + Total Bwd packets == 1)")
print("="*70)

iat_bad = df["Flow IAT Mean"].isna()
print(f"\nRows with NaN Flow IAT Mean: {iat_bad.sum():>4}")

if iat_bad.any():
    sample = df.loc[iat_bad, ["Total Fwd Packet", "Total Bwd packets", "Flow Duration"]]
    sample = sample.assign(total_packets=lambda x: x["Total Fwd Packet"] + x["Total Bwd packets"])
    print(f"\nTotal packet count distribution for those rows:")
    print(sample["total_packets"].value_counts().sort_index())
    print(f"\nFlow Duration for those rows:")
    print(sample["Flow Duration"].describe())

Hypothesis 1: Flow Bytes/s and Flow Packets/s break on the same rows
              (both denominators are flow_duration == 0)

Loaded Friday-WorkingHours.parquet: 547,915 rows

Rows with bad Flow Bytes/s:    348
Rows with bad Flow Packets/s:  348
Rows with EITHER bad:          348
Rows with BOTH bad:            348
Mismatch (XOR):                  0

Sample of bad rows on Friday — Flow Duration values:
count    348.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
Name: Flow Duration, dtype: float64


Hypothesis 2: Flow IAT NaN rows are single-packet flows
              (i.e. Total Fwd Packet + Total Bwd packets == 1)

Rows with NaN Flow IAT Mean:   52

Total packet count distribution for those rows:
total_packets
1    52
Name: count, dtype: int64

Flow Duration for those rows:
count    52.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: Flow Duration, dtype: float64


## Observations from NaN/Inf audit

**Total problem cells: 1,610 NaN + 984 Inf across 6 features in 2.1M rows
(approximately 0.001% of cells).**

### Affected features and structural cause

| Feature group | Cells affected | Mathematical cause |
|---|---|---|
| Flow Bytes/s | 793 (602 NaN + 191 Inf) | `bytes / duration` where `duration == 0` |
| Flow Packets/s | 793 (all Inf) | `packets / duration` where `duration == 0` |
| Flow IAT Mean/Std/Max/Min | 252 each (all NaN) | undefined for single-packet flows |

### Structural findings (verified empirically on Friday subset)

- Flow Bytes/s and Flow Packets/s break on **identical rows** because they 
  share the `Flow Duration` denominator. On Friday, 348 rows are affected; 
  XOR of the two bad-row masks is zero.
- Flow IAT NaN rows are **strictly single-packet flows** (52 of 52 on Friday 
  have exactly 1 total packet, all with `Flow Duration == 0`).
- Single-packet rows are a subset of zero-duration rows. Single-packet rows
  trigger all six feature failures; multi-packet zero-duration rows trigger 
  only the two flow-rate features.

### Cause

CICFlowMeter timestamps are microsecond-resolution. Flows where all packets 
arrive within a single microsecond are recorded with `Flow Duration == 0`, 
which produces undefined values in derived rate and IAT features. This is a 
known measurement limitation, not a tooling bug.

### Handling decision (Option A — drop rows)

Drop all rows with any NaN or Inf in numeric features before training. 
Affects approximately 0.06% of total rows (~1,200 unique rows out of 
2,100,814). Defensible because:

1. The affected rows represent measurement edge cases, not representative 
   flow behaviour.
2. The data loss is operationally negligible.
3. The alternative (imputation with arbitrary replacement values) introduces 
   methodological parameters that complicate the analysis.

To be implemented in `src/preprocessing/clean.py` (Week 2 work). Will be 
documented in Methodology chapter §4.3.

In [ ]:
"""Measure the impact of option A (drop rows with any NaN or Inf) across all days."""

total_rows = 0
total_clean = 0
total_dropped = 0
per_day_rows_dropped = {}

print(f"{'Day':<12} {'Rows':>10} {'Bad rows':>10} {'Drop %':>8} {'Bad rows benign':>18} {'Bad rows attack':>18}")
print("-" * 80)

for path in day_files:
    day = path.stem.replace("-WorkingHours", "")
    df = pd.read_parquet(path)
    
    numeric = df.select_dtypes(include="number")
    bad_rows = numeric.isna().any(axis=1) | numeric.apply(lambda c: np.isinf(c)).any(axis=1)
    n_bad = bad_rows.sum()
    
    bad_label_dist = df.loc[bad_rows, "Label"].value_counts()
    n_benign_bad = bad_label_dist.get("BENIGN", 0)
    n_attack_bad = n_bad - n_benign_bad
    
    per_day_rows_dropped[day] = n_bad
    total_rows += len(df)
    total_clean += len(df) - n_bad
    total_dropped += n_bad
    
    pct_drop = 100 * n_bad / len(df)
    print(f"{day:<12} {len(df):>10,} {n_bad:>10,} {pct_drop:>7.3f}% {n_benign_bad:>18,} {n_attack_bad:>18,}")

print("-" * 80)
print(f"{'TOTAL':<12} {total_rows:>10,} {total_dropped:>10,} {100*total_dropped/total_rows:>7.3f}%")
print(f"\nClean rows after Option A: {total_clean:,} ({100*total_clean/total_rows:.4f}% of original)")

Day                Rows   Bad rows   Drop %    Bad rows benign    Bad rows attack
--------------------------------------------------------------------------------
Friday          547,915        348   0.064%                220                128
Monday          371,749         60   0.016%                 60                  0
Thursday        362,368        291   0.080%                291                  0
Tuesday         322,003         23   0.007%                 23                  0
Wednesday       496,779         71   0.014%                 30                 41
--------------------------------------------------------------------------------
TOTAL         2,100,814        793   0.038%

Clean rows after Option A: 2,100,021 (99.9623% of original)


## Empirical confirmation of Option A handling

After applying Option A (drop rows with any NaN or Inf in numeric features) 
across the whole dataset:

- **Rows dropped: 793 (0.038% of 2,100,814).**
- **Rows retained: 2,100,021 (99.96%).**
- **Class balance preserved:** dropped rows are 624 benign + 169 attack 
  (78.7% benign / 21.3% attack), almost exactly mirroring the dataset's 
  global 78.91% / 21.09% ratio. The drop is class-neutral.

### Structural observation worth recording

The 793 affected rows are **not uniformly distributed across attack types**:

- **Tuesday and Thursday: zero attack rows affected.** Their attacks 
  (FTP/SSH-Patator brute force; web attacks; infiltration) are sustained 
  multi-packet sessions that always have measurable duration.
- **Wednesday and Friday: attack rows over-represented in bad rows** 
  relative to that day's attack rate. Their attacks (DoS Hulk floods, 
  PortScan probes, DDoS) include patterns that register as zero-duration 
  single-packet flows.

This implies zero-duration flows correlate with **scanning- and probe-style 
attack patterns**, not attacks generally. A future-work direction: keep 
these rows and add a binary indicator feature for "zero-duration flow" 
rather than dropping them, on the hypothesis that the indicator carries 
useful signal for some attack types. Not pursued here to keep the 
preprocessing pipeline minimal.

### Final preprocessing decision

`src/preprocessing/clean.py` (to be written in Week 2) will:
1. Read each Parquet file.
2. Drop rows where any numeric column has NaN or +/-Inf.
3. Persist a manifest of (rows_in, rows_out, n_dropped) for audit trail.

## Constant and near-constant features

Features carrying zero or negligible information are candidates for removal 
before training. Three categories:

- **Truly constant** — single unique value across all rows.
- **Effectively constant** — one value dominates ≥99.9% of rows.
- **Zero-variance** — all-zero columns (technically constant but worth 
  flagging separately, often indicates a feature that doesn't apply to 
  this dataset).

Identifying these now lets the preprocessing pipeline drop them at source, 
saving training time and reducing the risk of weight updates on 
information-free features.

In [6]:
"""Identify constant and near-constant features across the whole dataset."""

# Aggregate feature statistics across all five days
all_data = []
for path in day_files:
    df = pd.read_parquet(path)
    all_data.append(df)

full_df = pd.concat(all_data, ignore_index=True)
print(f"Combined: {len(full_df):,} rows × {full_df.shape[1]} columns")

# Numeric features only
numeric_df = full_df.select_dtypes(include="number")
print(f"Numeric features: {len(numeric_df.columns)}\n")

# Compute, for each column, the unique-value count and the fraction held
# by the most-common value (the "modal fraction")
feature_stats = []
for col in numeric_df.columns:
    series = numeric_df[col]
    n_unique = series.nunique(dropna=True)
    
    # Modal value and its share
    if n_unique > 0:
        value_counts = series.value_counts(dropna=True)
        modal_value = value_counts.index[0]
        modal_count = value_counts.iloc[0]
        modal_pct = 100 * modal_count / len(series)
    else:
        modal_value = None
        modal_count = 0
        modal_pct = 0.0
    
    feature_stats.append({
        "feature": col,
        "n_unique": n_unique,
        "modal_value": modal_value,
        "modal_pct": round(modal_pct, 3),
    })

stats_df = pd.DataFrame(feature_stats).sort_values("modal_pct", ascending=False)

# Categorise
truly_constant   = stats_df[stats_df["n_unique"] == 1]
nearly_constant  = stats_df[(stats_df["n_unique"] > 1) & (stats_df["modal_pct"] >= 99.9)]
informative      = stats_df[stats_df["modal_pct"] < 99.9]

print(f"{'='*70}")
print(f"Truly constant features (n_unique == 1):  {len(truly_constant)}")
print(f"{'='*70}")
if len(truly_constant) > 0:
    print(truly_constant[["feature", "modal_value"]].to_string(index=False))

print(f"\n{'='*70}")
print(f"Effectively constant features (modal value ≥ 99.9% of rows):  {len(nearly_constant)}")
print(f"{'='*70}")
if len(nearly_constant) > 0:
    print(nearly_constant[["feature", "n_unique", "modal_value", "modal_pct"]].to_string(index=False))

print(f"\n{'='*70}")
print(f"Informative features:  {len(informative)} (modal value < 99.9%)")
print(f"{'='*70}")
print("Top 10 most-skewed informative features (modal_pct just under 99.9%):")
print(informative.head(10)[["feature", "n_unique", "modal_pct"]].to_string(index=False))

Combined: 2,100,814 rows × 84 columns
Numeric features: 79

Truly constant features (n_unique == 1):  0

Effectively constant features (modal value ≥ 99.9% of rows):  6
            feature  n_unique  modal_value  modal_pct
      Bwd URG Flags         2          0.0    100.000
     URG Flag Count         5          0.0     99.996
      Fwd URG Flags         5          0.0     99.996
Subflow Bwd Packets         2          0.0     99.992
     ECE Flag Count         5          0.0     99.967
     CWR Flag Count         9          0.0     99.964

Informative features:  73 (modal value < 99.9%)
Top 10 most-skewed informative features (modal_pct just under 99.9%):
            feature  n_unique  modal_pct
Subflow Fwd Packets         2     97.369
Fwd Packet/Bulk Avg        98     95.859
  Fwd Bulk Rate Avg     27747     95.859
 Fwd Bytes/Bulk Avg      4570     95.859
         Active Std    210363     89.761
           Idle Std    217029     89.514
Bwd Packet/Bulk Avg       822     86.117
 Bwd B

## Observations from constant/near-constant feature audit

**Result: 0 truly constant features, 6 effectively constant (≥99.9% modal).**

### Drop list (6 features, all with modal value 0.0)

| Feature | Modal % | Rationale |
|---|---:|---|
| Bwd URG Flags | 100.000 | TCP URG flag rare in lab traffic |
| URG Flag Count | 99.996 | Total URG flags, same |
| Fwd URG Flags | 99.996 | Forward URG flags, same |
| Subflow Bwd Packets | 99.992 | Backward subflow packet count, almost always 0 |
| ECE Flag Count | 99.967 | ECN Echo flag, almost never used |
| CWR Flag Count | 99.964 | ECN Congestion Window Reduced flag, same |

These features will be dropped by the preprocessing pipeline 
(`src/preprocessing/clean.py`, Week 2 work).

**Justification for ≥99.9% threshold:** features with modal share above this 
level carry too little variation to support reliable model training and 
introduce sparse-feature numerical risk in neural network training. 
Dropping them simplifies the feature space without measurable information loss.

### Features to monitor but keep

- **Bulk transfer triplet** (`Fwd Packet/Bulk Avg`, `Fwd Bulk Rate Avg`, 
  `Fwd Bytes/Bulk Avg`): all 95.86% modal. Highly correlated by construction 
  (shared bulk-event denominator). The Pearson |r|>0.95 correlation filter 
  in preprocessing will likely drop 2 of the 3.
- **Active Std, Idle Std**: ~89% modal. Genuine signal in the non-zero tail.

### Working feature set

After dropping these 6, the working numeric feature set has **73 features** 
plus the categorical Label. Identifier columns (Flow ID, Src IP, Dst IP, 
Timestamp) will be excluded separately.

## Protocol = 0 artefact rows

CICFlowMeter occasionally records non-IP frames (ARP, STP, LLDP, mDNS) as 
flow records with `Protocol = 0` and `Src Port = Dst Port = 0`. Observed 
in notebook 01. Quantify here and decide handling.

Three questions:
1. **Count** — how many such rows exist in the full dataset?
2. **Label distribution** — are they all benign, or do any attack labels 
   appear?
3. **Overlap with NaN/Inf rows** — do the ones we're already dropping 
   include the artefact rows?

In [7]:
"""Quantify Protocol=0 artefact rows."""

# `full_df` is still in memory from the previous cell, but defensive
if "full_df" not in dir():
    all_data = []
    for path in day_files:
        all_data.append(pd.read_parquet(path))
    full_df = pd.concat(all_data, ignore_index=True)

print(f"Total rows in full dataset: {len(full_df):,}")

# Identify the artefact rows
proto_zero = full_df["Protocol"] == 0
src_port_zero = full_df["Src Port"] == 0
dst_port_zero = full_df["Dst Port"] == 0

# Three masks of progressive strictness
mask_proto_only      = proto_zero
mask_proto_and_ports = proto_zero & src_port_zero & dst_port_zero
mask_any_zero_port   = src_port_zero | dst_port_zero

print(f"\n{'='*70}")
print(f"Counts at three definitions of 'artefact'")
print(f"{'='*70}")
print(f"Protocol == 0 only:                       {mask_proto_only.sum():>8,}")
print(f"Protocol == 0 AND both ports == 0:        {mask_proto_and_ports.sum():>8,}")
print(f"Either source or dest port == 0:          {mask_any_zero_port.sum():>8,}")

# Use the strictest definition (the textbook artefact pattern) for the rest
print(f"\n{'='*70}")
print(f"Label distribution of strict artefact rows (Proto=0 + both ports=0)")
print(f"{'='*70}")
artefact_labels = full_df.loc[mask_proto_and_ports, "Label"].value_counts(dropna=False)
print(artefact_labels.head(20).to_string())

# Per-day distribution
print(f"\n{'='*70}")
print(f"Per-day distribution of strict artefact rows")
print(f"{'='*70}")
day_breakdown = []
for path in day_files:
    day = path.stem.replace("-WorkingHours", "")
    df = pd.read_parquet(path, columns=["Protocol", "Src Port", "Dst Port", "Label"])
    mask = (df["Protocol"] == 0) & (df["Src Port"] == 0) & (df["Dst Port"] == 0)
    day_breakdown.append({
        "day": day,
        "total_rows": len(df),
        "artefact_rows": mask.sum(),
        "artefact_pct": round(100 * mask.sum() / len(df), 3),
    })
day_breakdown_df = pd.DataFrame(day_breakdown)
print(day_breakdown_df.to_string(index=False))

Total rows in full dataset: 2,100,814

Counts at three definitions of 'artefact'
Protocol == 0 only:                          1,628
Protocol == 0 AND both ports == 0:           1,628
Either source or dest port == 0:             1,628

Label distribution of strict artefact rows (Proto=0 + both ports=0)
Label
BENIGN      1625
PortScan       3

Per-day distribution of strict artefact rows
      day  total_rows  artefact_rows  artefact_pct
   Friday      547915            303         0.055
   Monday      371749            316         0.085
 Thursday      362368            361         0.100
  Tuesday      322003            317         0.098
Wednesday      496779            331         0.067


In [8]:
"""Cross-check: overlap between Protocol=0 artefacts and NaN/Inf bad rows."""

# Recompute both masks on the full dataset
numeric_full = full_df.select_dtypes(include="number")
nan_inf_mask = numeric_full.isna().any(axis=1) | numeric_full.apply(np.isinf).any(axis=1)
artefact_mask = full_df["Protocol"] == 0

print(f"NaN/Inf bad rows:        {nan_inf_mask.sum():>6,}")
print(f"Protocol=0 artefacts:    {artefact_mask.sum():>6,}")
print(f"Both:                    {(nan_inf_mask & artefact_mask).sum():>6,}")
print(f"Either:                  {(nan_inf_mask | artefact_mask).sum():>6,}")
print(f"Only NaN/Inf:            {(nan_inf_mask & ~artefact_mask).sum():>6,}")
print(f"Only artefact:           {(~nan_inf_mask & artefact_mask).sum():>6,}")

print(f"\nIf we drop both populations:")
combined_drop = nan_inf_mask | artefact_mask
print(f"  Rows dropped: {combined_drop.sum():,} ({100*combined_drop.sum()/len(full_df):.3f}% of dataset)")
print(f"  Rows retained: {(~combined_drop).sum():,} ({100*(~combined_drop).sum()/len(full_df):.3f}%)")

NaN/Inf bad rows:           793
Protocol=0 artefacts:     1,628
Both:                         7
Either:                   2,414
Only NaN/Inf:               786
Only artefact:            1,621

If we drop both populations:
  Rows dropped: 2,414 (0.115% of dataset)
  Rows retained: 2,098,400 (99.885%)


## Observations from Protocol=0 artefact audit

**Result: 1,628 artefact rows (0.078% of dataset).**

### Structural cleanness

Three definitions of "artefact" tested — `Protocol=0` alone, `Protocol=0 AND 
both ports=0`, and `either port=0` — all yielded the **identical count of 
1,628**. The three conditions co-occur perfectly: every artefact row has 
`Protocol=0`, `Src Port=0`, AND `Dst Port=0`. This means the filter rule 
is unambiguous: **`Protocol == 0` alone catches all artefacts**.

### Label distribution

- 1,625 rows labelled `BENIGN`.
- 3 rows labelled `PortScan` — likely ARP/multicast side-effects of 
  scanning activity, captured by Engelen's labelling logic. Operationally 
  negligible (3 in 159,151 PortScan flows).

### Per-day distribution

Uniform across days (0.055%–0.100%), confirming these are background 
artefacts of the lab network's non-IP infrastructure traffic (ARP, STP, 
LLDP, mDNS), not attack-correlated.

### Overlap with NaN/Inf rows

- NaN/Inf bad rows: 793.
- Protocol=0 artefacts: 1,628.
- Intersection: only 7 rows.

The two populations are **essentially disjoint**. They have different 
structural causes (NaN/Inf = zero-duration flows; Protocol=0 = non-IP 
frames) and different filter rules independently catch them.

The dominance of "only artefact" (1,621) over "only NaN/Inf" (786) is 
informative: most non-IP frames produce sensible (zero-valued) numeric 
outputs from CICFlowMeter, rather than NaN/Inf. CICFlowMeter handles 
edge-case input more robustly than expected.

### Final preprocessing decision

`src/preprocessing/clean.py` (Week 2) will apply two independent filters:

1. **Drop rows with `Protocol == 0`** (1,628 rows, 0.078%).
2. **Drop rows with any NaN or +/-Inf in numeric features** (793 rows, 0.038%).
3. **Combined drop: 2,414 rows (0.115%). Retained: 99.885% of original.**

Both filters are documented in Methodology chapter §4.3 with the structural 
justifications above.

## Notebook 02 summary — what we learned

**Three data-integrity concerns audited and resolved.**

### Headline numbers

| Concern | Affected rows | % of dataset | Disposition |
|---|---:|---:|---|
| NaN or Inf in numeric features | 793 | 0.038% | Drop |
| Protocol = 0 artefact rows | 1,628 | 0.078% | Drop |
| Constant or near-constant features | 6 features | — | Drop columns |
| **Combined row drop** | **2,414** | **0.115%** | **Drop** |

### Working dataset after cleanup

- **Rows retained: 2,098,400 of 2,100,814 (99.885%).**
- **Numeric features retained: 73 of 79 (after dropping the 6 near-constant).**
- **Identifier columns excluded from training: 4** (Flow ID, Src IP, Dst IP, 
  Timestamp — kept aside for chronological analysis).
- **Label column: 1**, retained as classification target.

### Methodological implications

1. **Engelen's cleaning is high-quality** but does not eliminate all edge 
   cases. The remaining issues are mathematically-driven (zero-duration 
   flows) or structural (non-IP frames CICFlowMeter records with zero 
   protocol/ports), not data corruption.
2. **The cleanup is class-neutral.** Dropped rows mirror the global 
   benign/attack ratio. No micro-class is harmed.
3. **The drop rules are independent and unambiguous** — the two row-level 
   filters and the column-level constant filter compose cleanly into a 
   minimal preprocessing pipeline.

### Pre-processing module specification (for Week 2 work)

`src/preprocessing/clean.py` shall:
- Accept a Parquet file path.
- Drop rows where `Protocol == 0`.
- Drop rows where any numeric feature is NaN or +/-Inf.
- Drop the 6 near-constant feature columns:
  `Bwd URG Flags`, `URG Flag Count`, `Fwd URG Flags`, 
  `Subflow Bwd Packets`, `ECE Flag Count`, `CWR Flag Count`.
- Persist a manifest documenting (rows_in, rows_out, n_dropped_proto0, 
  n_dropped_nan_inf, n_columns_in, n_columns_out).
- Be deterministic — same input always produces same output.

### Open questions deferred to later notebooks

- Numeric feature distributions and outliers (notebook 03).
- Attack timeline and per-day attack rates (notebook 03 or 04).
- Pearson correlation matrix and the Pearson |r| > 0.95 filter for the 
  bulk-rate triplet and similar groups (notebook 04 or directly in the 
  preprocessing pipeline).

## Methodological refinement — added after notebook 04 (CIC-UNSW-NB15) inspection

The 6-column near-constant drop list above was derived from CIC-IDS2017 alone.
After acquiring CIC-UNSW-NB15 (notebook 04) and verifying the same columns
on the secondary dataset, i found one column behaving differently:

| Column | CIC-IDS2017 modal % | CIC-UNSW-NB15 modal % | n_unique on UNSW |
|---|---:|---:|---:|
| Bwd URG Flags | 100.000 | 100.000 | 1 |
| URG Flag Count | 99.996 | 100.000 | 1 |
| Fwd URG Flags | 99.996 | 100.000 | 1 |
| **Subflow Bwd Packets** | **99.992** | **97.035** | **551** |
| ECE Flag Count | 99.967 | 100.000 | 1 |
| CWR Flag Count | 99.964 | 100.000 | 1 |

`Subflow Bwd Packets` is near-constant on CIC-IDS2017 (only 2 unique values)
but informative on CIC-UNSW-NB15 (551 unique values, only 97% modal).
Dropping it on CIC-IDS2017 only would create a feature-space mismatch in
the cross-dataset evaluation. Dropping it on both would discard genuine signal
on UNSW.

**My Decision: keep `Subflow Bwd Packets` in both datasets.** The five
remaining columns (Bwd URG Flags, URG Flag Count, Fwd URG Flags, ECE
Flag Count, CWR Flag Count) are all truly constant in both datasets
and remain on the drop list.

Final feature-set count after cleaning:
- **CIC-IDS2017: 79 columns** (was 78 with the original 6-column drop)
- **CIC-UNSW-NB15: 76 columns** (no identifier columns to drop, only the
  same 5 near-constant columns)

Cross-dataset comparison still requires aligning to the 76-column shared
subset (notebook 04, observation 1). The 3-column difference is due to
CIC-IDS2017 keeping `Src Port`, `Dst Port`, `Protocol` while UNSW does
not; i will drop these from CIC-IDS2017 prior to model training.

This refinement is recorded in `src/preprocessing/clean.py` as a comment
on `NEAR_CONSTANT_COLS` and will be discussed in my Methodology chapter 4.3.